In [ ]:
# =============================================================================
# CELL 1 — Mount Drive and set working directory
# =============================================================================
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

PROJECT = '/content/drive/MyDrive/CASS_QESC'  # adjust if your folder name differs
os.chdir(PROJECT)

print("Working directory:", os.getcwd())
print("Files found:", os.listdir('.'))

In [ ]:
# =============================================================================
# CELL 2 — Install dependencies
# =============================================================================
# Uncomment first time only:
!pip install sentence-transformers numpy -q

In [ ]:
# =============================================================================
# CELL 3 — Load model (shared with Instantiation 1)
# =============================================================================
# Loads only the sentence encoder — no corpus yet.
# The same MiniLM model serves both instantiations.

import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded.")


# =============================================================================
# CELL 4 — Load PhonEx corpus and generate embeddings
# =============================================================================
# PhonEx: 100 phonics exercises for French-speaking dyslexic children aged 6-9.
# 5 error types × 5 difficulty levels × 4 entries per combination.
# Description field is encoded — written to match parent/teacher free-text queries.
#
# Run this cell once. It saves phonex_embeddings_v1.0.npy to your Drive folder.
# Re-run only if you update PHONICS_CORPUS_v1.0.json.
#
# Expected output:
#   PhonEx corpus: 100 entries
#   Embeddings shape: (100, 384)

import json

with open('PHONICS_CORPUS_v1.0.json', encoding='utf-8') as f:
    phonex = json.load(f)

print(f"PhonEx corpus: {len(phonex)} entries")

from collections import Counter
print(f"ERR_TYPE distribution: {dict(Counter(e['ERR_TYPE'] for e in phonex))}")
print(f"DIFF distribution: {dict(sorted(Counter(e['DIFF'] for e in phonex).items()))}")

# Encode description field
phonex_embeddings = model.encode(
    [e['description'] for e in phonex],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

np.save('phonex_embeddings_v1.0.npy', phonex_embeddings)
print(f"\nSaved phonex_embeddings_v1.0.npy")
print(f"Shape: {phonex_embeddings.shape}")


In [ ]:
# =============================================================================
# CELL 5 — Constraint Functions (Instantiation 2)
# =============================================================================
# C_DIFF: Gaussian — phonological difficulty alignment (ordinal, scale 1-5)
# C_ERR:  Categorical — error type alignment (nominal, 5 categories)
#
# Key difference from Instantiation 1:
#   C_ERR is CATEGORICAL not Gaussian — error types are nominal not ordinal.
#   A visual_confusion exercise is not "between" vowel_substitution and
#   letter_reversal — categories have no meaningful numeric distance.

def C_DIFF(diff_q: float, diff_c: float, lam: float = 0.5) -> float:
    """
    Gaussian Phonological Difficulty Alignment.
    Penalises mismatch between child's difficulty level and exercise level.
    DIFF scale: 1 (simplest) to 5 (most advanced).
    λ=0.5 matches Instantiation 1 for comparability.
    """
    return float(np.exp(-lam * (diff_q - diff_c) ** 2))


def C_ERR(err_q: str, err_c: str) -> float:
    """
    Categorical Error Type Alignment.
    1.0 = exact match (same primary error type)
    0.5 = partial overlap (errors sharing underlying phonological mechanism)
    0.0 = no match

    Partial overlap pairs — based on shared phonological processing demands:
      visual_confusion  ↔ letter_reversal     (both involve letter-level processing)
      blending_difficulty ↔ syllable_omission  (both involve syllable-level processing)
      blending_difficulty ↔ vowel_substitution (both involve phoneme-grapheme mapping)
    """
    if err_q == err_c:
        return 1.0
    partial_pairs = [
        {'visual_confusion', 'letter_reversal'},
        {'blending_difficulty', 'syllable_omission'},
        {'blending_difficulty', 'vowel_substitution'},
    ]
    if {err_q, err_c} in partial_pairs:
        return 0.5
    return 0.0

print("C_DIFF and C_ERR defined.")

# Quick verification
print("\nC_ERR spot checks:")
print(f"  visual_confusion vs visual_confusion = {C_ERR('visual_confusion','visual_confusion')}")
print(f"  visual_confusion vs letter_reversal  = {C_ERR('visual_confusion','letter_reversal')}")
print(f"  visual_confusion vs vowel_substitution = {C_ERR('visual_confusion','vowel_substitution')}")

print("\nC_DIFF spot checks (λ=0.5):")
for diff_c in [1, 2, 3, 4, 5]:
    print(f"  query DIFF=2 vs corpus DIFF={diff_c}: {C_DIFF(2, diff_c):.3f}")


# =============================================================================
# CELL 6 — Detection Functions (DIFF and ERR from free text)
# =============================================================================
# Rule-based classifiers — no training required.
# detect_DIFF: estimates child's current difficulty level from description.
# detect_ERR:  identifies primary error type from description keywords.
# Supports French, English, and Darija markers.

def detect_DIFF(text: str) -> float:
    """
    Estimates phonological difficulty level (1-5) from parent/teacher description.
    1 = isolated letters/sounds
    2 = simple words (CVC, two syllables)
    3 = two-syllable words, short sentences
    4 = sentences, reading aloud, connected text
    5 = paragraphs, fluency, comprehension, academic vocabulary
    """
    text_lower = text.lower()
    score = 2.5

    level5 = [
        'paragraphe', 'texte long', 'fluence', 'vitesse de lecture',
        'compréhension', 'texte académique', 'texte entier', 'lecture courante',
        'fluency', 'extended text', 'grade level', 'passage', 'comprehension',
        'lecture fluide', 'academic'
    ]
    level4 = [
        'phrase', 'phrases', 'sentence', 'sentences', 'connected text',
        'lire à voix haute', 'audience', 'lecture en classe',
        'reading aloud', 'paragraphe court'
    ]
    level3 = [
        'deux syllabes', 'polysyllabique', 'mots longs', 'mots de 3 syllabes',
        'two syllable', 'longer words', 'three syllable', 'mot long'
    ]
    level2 = [
        'mot simple', 'simple word', 'short word', 'cvc', 'mot court',
        'une syllabe', 'one syllable', 'mot de deux syllabes'
    ]
    level1 = [
        'lettre isolée', 'syllabe simple', 'son isolé', 'isolated letter',
        'single letter', 'lettre seule', 'simple syllable', 'letter alone'
    ]

    for m in level5:
        if m in text_lower: score += 1.2
    for m in level4:
        if m in text_lower: score += 0.7
    for m in level3:
        if m in text_lower: score += 0.3
    for m in level2:
        if m in text_lower: score -= 0.3
    for m in level1:
        if m in text_lower: score -= 0.8

    return float(min(max(round(score), 1), 5))


def detect_ERR(text: str) -> str:
    """
    Detects primary error type from parent/teacher free-text description.
    Returns one of: visual_confusion, vowel_substitution, syllable_omission,
                    letter_reversal, blending_difficulty
    Defaults to blending_difficulty if no clear signal found.
    Supports French, English, Darija markers.
    """
    text_lower = text.lower()

    patterns = {
        'visual_confusion': [
            'confond b et d', 'confuses b and d', 'b comme d', 'd comme b',
            'confond les lettres', 'miroir', 'mirror letters', 'b/d',
            'confond p et q', 'p/q', 'lit b pour d', 'reads b as d',
            'mélange b', 'mélange d', 'inverse b', 'inverse d',
            # Darija
            'kaykhelt bin b w d', 'ma yfarekch bin les lettres'
        ],
        'vowel_substitution': [
            'voyelle', 'vowel', 'confond les sons des voyelles',
            'o comme ou', 'ou comme o', 'lit mal les voyelles',
            'remplace la voyelle', 'substitue la voyelle',
            'mauvaise voyelle', 'wrong vowel', 'vowel error',
            'é comme è', 'ai comme ei', 'son de voyelle',
            # Darija
            'kaykhelt les voyelles', 'ma ysahbch les voyelles'
        ],
        'syllable_omission': [
            'syllabe', 'syllable', 'saute', 'omet', 'omits', 'drops',
            'mange des syllabes', 'skips syllables', 'raccourcit les mots',
            'mot trop court', 'word too short', 'partie du mot',
            'perd des syllabes', 'loses syllables', 'syllabe manquante',
            'missing syllable',
            # Darija
            'kaysal les syllabes', 'katkhelt les syllabes',
            'kaydel partie du mot', 'ma iqrach les syllabes mezyan'
        ],
        'letter_reversal': [
            'inverse les lettres', 'reverses letters', 'transpose',
            'dans le mauvais ordre', 'wrong order', 'retourne les lettres',
            'lit à l\'envers', 'reads backwards', 'intervertit',
            'lettres dans le mauvais ordre', 'letters out of order',
            'anagramme', 'anagram',
            'backwards', 'à l\'envers', 'lit à reculons',
            'dans le sens inverse', 'reverse', 'reverses words',
            'lit les mots à l\'envers', 'reads words backwards',
            'kayqlib', 'kayqra men liser'
        ],
        'blending_difficulty': [
            'ne peut pas fusionner', 'cannot blend', 'décode lettre par lettre',
            'letter by letter', 'son par son', 'ne fusionne pas',
            'sons séparés', 'separate sounds', 'épelle sans lire',
            'cannot read words', 'ne lit pas les mots', 'assemble pas les sons',
            'slow reader', 'lit très lentement', 'déchiffre lentement',
            'sounds out every letter', 'épelle chaque lettre',
            # Darija
            'ma iqrach', 'ma kayqrach mezyan', 'kayqra harf harf',
            'betti ma tqrach', 'weldi ma iqrach'
        ],
    }

    scores = {err: 0 for err in patterns}
    for err, markers in patterns.items():
        for m in markers:
            if m in text_lower:
                scores[err] += 1

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'blending_difficulty'

print("detect_DIFF and detect_ERR defined.")

# Quick test
test_queries = [
    "My child confuses b and d when reading simple words",
    "Elle saute des syllabes dans les mots longs",
    "Weldi ma iqrach mezyan, kayqra harf harf",
    "She reads sentences slowly losing the meaning",
]
print("\nDetection test:")
for q in test_queries:
    print(f"  '{q[:50]}...' → DIFF={detect_DIFF(q)}, ERR={detect_ERR(q)}")



In [ ]:
# =============================================================================
# CELL 7 — Optimized parameters (from Dirichlet weight search on validation set)
# =============================================================================
# MHR = 1.780 / 3.000 on 20-query validation set
# α=0.412 (cosine), β1=0.461 (C_DIFF), β2=0.127 (C_ERR), λ=0.5


""" best configuration n1
ALPHA_P  = 0.3047
BETA1_P  = 0.1855
BETA2_P  = 0.5099
LAM_P    = 0.2
THRESHOLD_P = 0.35
"""

#  best configuration n2
#α=0.635, β1=0.160, β2=0.205, λ=0.5
#α=0.619, β1=0.082, β2=0.299, λ=1.0

ALPHA_P  = 0.3047
BETA1_P  = 0.1855
BETA2_P  = 0.5099
LAM_P    = 0.2
THRESHOLD_P = 0.35


print(f"Parameters: α={ALPHA_P}, β1={BETA1_P}, β2={BETA2_P}, λ={LAM_P}")
print(f"Check α+β1+β2 = {ALPHA_P+BETA1_P+BETA2_P:.6f}")


# =============================================================================
# CELL 8 — CASS Retrieval Function (Instantiation 2)
# =============================================================================

def retrieve_phonex(query: str, top_k: int = 1, verbose: bool = True):
    """
    CASS retrieval for PhonEx corpus.

    Formula:
      CASS(q,c) = α·cos(q,c) + β1·C_DIFF(q,c) + β2·C_ERR(q,c)

    Args:
        query:   parent/teacher free-text description of child's reading difficulty
        top_k:   number of results to return
        verbose: print scoring table

    Returns:
        list of (entry, cass_score) tuples
    """
    # Step 1 — encode query
    qvec = model.encode([query], normalize_embeddings=True)[0]

    # Step 2 — cosine similarity
    cos_scores = phonex_embeddings @ qvec

    # Step 3 — detect DIFF and ERR from query
    diff_q = detect_DIFF(query)
    err_q  = detect_ERR(query)

    if verbose:
        print(f"Query: '{query}'")
        print(f"Detected DIFF: {diff_q} | Detected ERR: {err_q}")
        print()

    # Step 4 — compute CASS for each entry
    results = []
    for i, entry in enumerate(phonex):
        cos    = float(cos_scores[i])
        c_diff = C_DIFF(diff_q, entry['DIFF'], lam=LAM_P)
        c_err  = C_ERR(err_q, entry['ERR_TYPE'])
        cass   = ALPHA_P * cos + BETA1_P * c_diff + BETA2_P * c_err
        results.append((entry, cass, cos, c_diff, c_err))

    # Step 5 — sort by CASS score
    results.sort(key=lambda x: x[1], reverse=True)

    if verbose:
        print(f"{'Rank':<5} {'ID':<16} {'ERR_TYPE':<22} {'D':>2} "
              f"{'CASS':>6} {'cos':>6} {'C_DIFF':>6} {'C_ERR':>6}")
        print("-" * 75)
        for rank, (e, cass, cos, c_diff, c_err) in enumerate(results[:10], 1):
            print(f"{rank:<5} {e['id']:<16} {e['ERR_TYPE']:<22} {e['DIFF']:>2} "
                  f"{cass:>6.3f} {cos:>6.3f} {c_diff:>6.3f} {c_err:>6.3f}")
        print()

    # Step 6 — threshold check
    if results[0][1] < THRESHOLD_P:
        if verbose:
            print(f"Score {results[0][1]:.3f} below threshold. Ask to rephrase.")
        return []

    # Step 7 — return results
    top = results[0]
    if verbose:
        print(f"TOP MATCH: {top[0]['id']} — {top[0]['ERR_TYPE']} DIFF={top[0]['DIFF']}")
        print(f"Desc: {top[0]['description'][:120]}...")
        print(f"Instruction: {top[0]['child_instruction']}")

    return [(e, cass) for e, cass, *_ in results[:top_k]]

print("retrieve_phonex() defined.")


# =============================================================================
# CELL 9 — Demo: test queries
# =============================================================================

test_queries_phonex = [
    # Category A — direct technical vocabulary (French)
    "Mon enfant confond b et d quand il lit des mots simples",
    "Elle ne peut pas fusionner les sons pour lire un mot",
    "Il saute des syllabes dans les mots de trois syllabes",

    # Category A — direct technical vocabulary (English)
    "My child confuses b and d when reading simple words",
    "Child reads letter by letter and cannot blend sounds together",

    # Category B — indirect descriptions
    "My daughter reads bateau as dateau and sometimes skips the middle part",
    "He reads every word backwards and gets very frustrated",
    "She reads sentences very slowly and loses the meaning completely",
    "Mon fils lit très lentement, il s'arrête sur chaque lettre",

    # Category C — multilingual Darija/code-switching
    "Weldi ma iqrach mezyan, kayqra harf harf",
    "Benti katkhelt bin les syllabes, katsal les mots",
    "Ma iqrach les mots, kaykhelt les lettres",
]

for query in test_queries_phonex:
    print("=" * 70)
    retrieve_phonex(query, top_k=1, verbose=True)
    print()



In [ ]:
# =============================================================================
# CELL 10 — C_DIFF and C_ERR Safety Gate Demonstration (PhonEx)
# =============================================================================
# Shows that pedagogically dangerous exercises — wrong difficulty level or
# wrong error type — are suppressed by CASS even when cosine similarity ranks
# them high.
#
# C_DIFF dangerous: |DIFF(q) - DIFF(c)| >= 2
#   e.g. a DIFF=1 query attracting a DIFF=5 paragraph-level exercise
#
# C_ERR dangerous: C_ERR(q,c) = 0.0 (completely wrong error type)
#   e.g. a visual_confusion query attracting a vowel_substitution exercise
#
# Requires: phonex, phonex_embeddings, model, C_DIFF, C_ERR,
#           ALPHA_P, BETA1_P, BETA2_P, LAM_P from Cells 4-7
# =============================================================================

# ── QUERY SETS ────────────────────────────────────────────────────────────────

# 3 queries testing C_DIFF safety gate
# Chosen to have extreme DIFF values (1 or 5) where dangerous entries exist
cdiff_queries = [
    # Low-DIFF queries — should NOT retrieve DIFF=4 or 5 exercises
    ("Child cannot distinguish b from d even with isolated letters",
     1.0, "visual_confusion"),
    ("Ma fille ne sait pas fusionner deux sons simples comme 'la' et 'pa'",
     1.0, "blending_difficulty"),
    # High-DIFF query — should NOT retrieve DIFF=1 or 2 exercises
    ("Child reads long paragraphs but completely loses the meaning while decoding",
     5.0, "blending_difficulty"),
]

# 3 queries testing C_ERR safety gate
# Chosen to surface wrong-error-type entries with high cosine
cerr_queries = [
    ("My child confuses b and d in every word he reads",
     2.0, "visual_confusion"),
    ("Elle saute systématiquement des syllabes dans les mots longs",
     3.0, "syllable_omission"),
    ("Child reads letter by letter and cannot blend any sounds together",
     2.0, "blending_difficulty"),
]

# ── HELPER ────────────────────────────────────────────────────────────────────

def get_rankings(query, diff_q, err_q):
    """Returns (cass_sorted, cos_sorted) for a query with given DIFF and ERR."""
    qvec = model.encode([query], normalize_embeddings=True)[0]
    cos_scores = phonex_embeddings @ qvec
    results = []
    for i, e in enumerate(phonex):
        cos    = float(cos_scores[i])
        c_diff = C_DIFF(diff_q, e['DIFF'], lam=LAM_P)
        c_err  = C_ERR(err_q, e['ERR_TYPE'])
        cass   = ALPHA_P * cos + BETA1_P * c_diff + BETA2_P * c_err
        results.append((e, cass, cos, c_diff, c_err))
    cass_sorted = sorted(results, key=lambda x: x[1], reverse=True)
    cos_sorted  = sorted(results, key=lambda x: x[2], reverse=True)
    return cass_sorted, cos_sorted

def cass_rank_of(entry_id, cass_sorted):
    return next(r+1 for r,(e,_,_,_,_) in enumerate(cass_sorted) if e['id']==entry_id)

# ── C_DIFF SAFETY GATE ───────────────────────────────────────────────────────

print("C_DIFF SAFETY GATE DEMONSTRATION")
print("Dangerous threshold: |DIFF(q) - DIFF(c)| >= 2")
print("="*75)

all_cdiff_drops = []

for query, diff_q, err_q in cdiff_queries:
    cass_sorted, cos_sorted = get_rankings(query, diff_q, err_q)

    dangerous = []
    for rank, (e, cass, cos, c_diff, c_err) in enumerate(cos_sorted[:20]):
        if abs(diff_q - e['DIFF']) >= 2:
            cr = cass_rank_of(e['id'], cass_sorted)
            drop = cr - (rank+1)
            dangerous.append((rank+1, cr, e, cos, c_diff, drop))

    if dangerous:
        print(f"\nQuery: '{query[:60]}'")
        print(f"DIFF={diff_q} | ERR={err_q}")
        print(f"{'Entry':<16} {'D':>2} {'cos':>6} {'C_DIFF':>6} "
              f"{'cos#':>5} {'cass#':>6} {'effect'}")
        print("-"*60)
        for cos_r, cass_r, e, cos, c_diff, drop in dangerous[:5]:
            drop_str = f"↓ {drop} ranks" if drop > 0 else "unchanged"
            print(f"{e['id']:<16} {e['DIFF']:>2} {cos:>6.3f} {c_diff:>6.3f} "
                  f"{cos_r:>5} {cass_r:>6}  {drop_str}")
            all_cdiff_drops.append(drop)

        # Show what CASS correctly returns instead
        top = cass_sorted[0]
        print(f"\nCASS top match: {top[0]['id']} DIFF={top[0]['DIFF']} "
              f"ERR={top[0]['ERR_TYPE']} — CASS={top[1]:.3f}")

if all_cdiff_drops:
    print(f"\nC_DIFF mean suppression: {sum(all_cdiff_drops)/len(all_cdiff_drops):.1f} positions")
    print(f"C_DIFF max suppression:  {max(all_cdiff_drops)} positions")

# ── C_ERR SAFETY GATE ────────────────────────────────────────────────────────

print("\n\nC_ERR SAFETY GATE DEMONSTRATION")
print("Dangerous threshold: C_ERR(q,c) = 0.0 (wrong error type, no shared mechanism)")
print("="*75)

all_cerr_drops = []

for query, diff_q, err_q in cerr_queries:
    cass_sorted, cos_sorted = get_rankings(query, diff_q, err_q)

    dangerous = []
    for rank, (e, cass, cos, c_diff, c_err) in enumerate(cos_sorted[:20]):
        if c_err == 0.0:
            cr = cass_rank_of(e['id'], cass_sorted)
            drop = cr - (rank+1)
            dangerous.append((rank+1, cr, e, cos, c_err, drop))

    if dangerous:
        print(f"\nQuery: '{query[:60]}'")
        print(f"DIFF={diff_q} | ERR={err_q}")
        print(f"{'Entry':<16} {'ERR_TYPE':<22} {'cos':>6} {'C_ERR':>6} "
              f"{'cos#':>5} {'cass#':>6} {'effect'}")
        print("-"*70)
        for cos_r, cass_r, e, cos, c_err, drop in dangerous[:5]:
            drop_str = f"↓ {drop} ranks" if drop > 0 else "unchanged"
            print(f"{e['id']:<16} {e['ERR_TYPE']:<22} {cos:>6.3f} {c_err:>6.1f} "
                  f"{cos_r:>5} {cass_r:>6}  {drop_str}")
            all_cerr_drops.append(drop)

        top = cass_sorted[0]
        print(f"\nCASS top match: {top[0]['id']} ERR={top[0]['ERR_TYPE']} "
              f"DIFF={top[0]['DIFF']} — CASS={top[1]:.3f}")

if all_cerr_drops:
    print(f"\nC_ERR mean suppression: {sum(all_cerr_drops)/len(all_cerr_drops):.1f} positions")
    print(f"C_ERR max suppression:  {max(all_cerr_drops)} positions")

print("\n\nRun this cell after Cell 8 (retrieve_phonex defined).")
print("Share results to generate Table 6 and Table 7 for Section 5.")

In [ ]:
# =============================================================================
# CELL 11 — Dirichlet Weight Search for PhonEx (Instantiation 2)
# =============================================================================
# Optimizes α, β1, β2, λ for PhonEx corpus using a 20-query validation set.
# Same procedure as Instantiation 1 (Bergstra & Bengio 2012).
#
# Requires: phonex, phonex_embeddings, model, C_DIFF, C_ERR from Cells 4-5
# Output: best_phonex_params.json saved to Drive folder
# Runtime: ~2 minutes on Colab CPU
# =============================================================================

import itertools
import numpy as np
import json

# ── SEARCH CONFIG ─────────────────────────────────────────────────────────────
N_WEIGHT_SAMPLES = 200
RANDOM_SEED      = 42
LAMBDA_VALUES    = [0.2, 0.5, 1.0, 1.5, 2.0]
# Note: C_ERR is categorical — no μ parameter needed.
# We optimize λ for C_DIFF only.

# ── VALIDATION SET ────────────────────────────────────────────────────────────
# 20 parent/teacher queries with annotated best exercise match.
# Format: (query, best_match_id, [acceptable_alternative_ids])
# Score: best=3, acceptable=2, other=1. MHR = mean score.
#
# Review and adjust expected matches based on your expert judgment
# before running — these are your ground truth labels.

VALIDATION_SET = [
    # Direct French — visual confusion
    ("Mon enfant confond b et d quand il lit des mots simples",
     "ex_vc_003", ["ex_vc_012", "ex_vc_011"]),

    # Direct French — blending
    ("Elle ne peut pas fusionner les sons pour lire un mot",
     "ex_bd_003", ["ex_bd_001", "ex_bd_012"]),

    # Direct French — syllable omission
    ("Il saute des syllabes dans les mots de trois syllabes",
     "ex_so_003", ["ex_so_012", "ex_so_005"]),

    # Direct English — visual confusion
    ("My child confuses b and d when reading simple words",
     "ex_vc_003", ["ex_vc_012", "ex_vc_002"]),

    # Direct English — blending
    ("Child reads letter by letter and cannot blend sounds together",
     "ex_bd_005", ["ex_bd_013", "ex_bd_003"]),

    # Indirect — syllable omission
    ("My daughter reads bateau as dateau and skips the middle part",
     "ex_so_003", ["ex_so_001", "ex_so_012"]),

    # Indirect — letter reversal
    ("He reads every word backwards and gets very frustrated",
     "ex_lr_001", ["ex_lr_010", "ex_lr_011"]),

    # Indirect — blending at sentence level
    ("She reads sentences very slowly and loses the meaning completely",
     "ex_bd_007", ["ex_bd_016", "ex_bd_019"]),

    # Indirect — visual confusion at word level
    ("Every time he sees the letter b he writes d instead",
     "ex_vc_001", ["ex_vc_011", "ex_vc_002"]),

    # Indirect — vowel substitution
    ("She reads 'mou' instead of 'mot' and confuses similar vowel sounds",
     "ex_vs_001", ["ex_vs_002", "ex_vs_010"]),

    # Difficulty level specific — DIFF=1
    ("Child cannot distinguish b from d even with isolated letters",
     "ex_vc_001", ["ex_vc_002", "ex_vc_010"]),

    # Difficulty level specific — DIFF=4/5
    ("Child reads paragraphs but completely loses meaning while decoding",
     "ex_bd_009", ["ex_bd_007", "ex_bd_018"]),

    # Difficulty level specific — DIFF=3
    ("Mon fils lit correctement les mots seuls mais pas dans les phrases",
     "ex_bd_005", ["ex_so_006", "ex_bd_007"]),

    # Multilingual Darija — blending
    ("Weldi ma iqrach mezyan, kayqra harf harf",
     "ex_bd_001", ["ex_bd_003", "ex_bd_011"]),

    # Multilingual Darija — syllable omission
    ("Benti katkhelt bin les syllabes, katsal les mots",
     "ex_so_001", ["ex_so_003", "ex_so_011"]),

    # Multilingual Darija — letter confusion
    ("Ma iqrach les mots, kaykhelt les lettres",
     "ex_vc_001", ["ex_bd_001", "ex_lr_001"]),

    # Mixed error description
    ("My child skips syllables and also confuses b and d",
     "ex_mx_001", ["ex_so_003", "ex_vc_003"]),

    # Advanced — DIFF=4 specific
    ("Il lit à voix haute devant la classe mais fait beaucoup d'erreurs",
     "ex_so_016", ["ex_bd_007", "ex_vc_007"]),

    # Vowel specific — French
    ("Elle lit 'tou' au lieu de 'tu' et confond les voyelles proches",
     "ex_vs_004", ["ex_vs_001", "ex_vs_013"]),

    # Letter reversal specific
    ("Child writes and reads letters in the wrong order constantly",
     "ex_lr_001", ["ex_lr_010", "ex_lr_003"]),
]

# ── HAND-ANNOTATED DIFF AND ERR FOR VALIDATION QUERIES ───────────────────────
# Pre-annotated to avoid running detect_DIFF/ERR 4000× (too slow)
# Must match order of VALIDATION_SET exactly

VAL_DIFF = [
    2.0,  # Q1  mots simples
    2.0,  # Q2  fusionner les sons
    2.0,  # Q3  mots de trois syllabes
    2.0,  # Q4  simple words
    2.0,  # Q5  letter by letter
    2.0,  # Q6  bateau/dateau
    2.0,  # Q7  reads backwards
    4.0,  # Q8  sentences slowly
    2.0,  # Q9  letter b/d
    2.0,  # Q10 vowel sounds
    1.0,  # Q11 isolated letters
    5.0,  # Q12 paragraphs
    3.0,  # Q13 mots seuls/phrases
    2.0,  # Q14 Darija harf harf
    2.0,  # Q15 Darija syllabes
    2.0,  # Q16 Darija lettres
    2.0,  # Q17 mixed error
    4.0,  # Q18 voix haute classe
    2.0,  # Q19 tou/tu
    2.0,  # Q20 wrong order
]

VAL_ERR = [
    'visual_confusion',    # Q1
    'blending_difficulty', # Q2
    'syllable_omission',   # Q3
    'visual_confusion',    # Q4
    'blending_difficulty', # Q5
    'syllable_omission',   # Q6
    'letter_reversal',     # Q7
    'blending_difficulty', # Q8
    'visual_confusion',    # Q9
    'vowel_substitution',  # Q10
    'visual_confusion',    # Q11
    'blending_difficulty', # Q12
    'blending_difficulty', # Q13
    'blending_difficulty', # Q14
    'syllable_omission',   # Q15
    'visual_confusion',    # Q16
    'syllable_omission',   # Q17
    'syllable_omission',   # Q18
    'vowel_substitution',  # Q19
    'letter_reversal',     # Q20
]

assert len(VAL_DIFF) == len(VALIDATION_SET)
assert len(VAL_ERR)  == len(VALIDATION_SET)

# ── PRE-COMPUTE ───────────────────────────────────────────────────────────────
print(f"Encoding {len(VALIDATION_SET)} validation queries...")
val_queries = [v[0] for v in VALIDATION_SET]
val_best    = [v[1] for v in VALIDATION_SET]
val_alts    = [v[2] for v in VALIDATION_SET]

val_vecs   = model.encode(val_queries, normalize_embeddings=True)
cos_matrix = val_vecs @ phonex_embeddings.T  # (20, 100)

diff_array = np.array([e['DIFF'] for e in phonex], dtype=float)  # (100,)

print(f"  Done. Cosine matrix: {cos_matrix.shape}")
print()

# ── SCORING ───────────────────────────────────────────────────────────────────
def compute_mhr_phonex(alpha, beta1, beta2, lam):
    scores = []
    for i in range(len(VALIDATION_SET)):
        cos    = cos_matrix[i]                                    # (100,)
        c_diff = np.exp(-lam * (VAL_DIFF[i] - diff_array) ** 2) # (100,)
        c_err  = np.array([C_ERR(VAL_ERR[i], e['ERR_TYPE'])
                           for e in phonex])                      # (100,)
        cass   = alpha * cos + beta1 * c_diff + beta2 * c_err    # (100,)

        top_id = phonex[int(np.argmax(cass))]['id']

        if top_id == val_best[i]:
            scores.append(3)
        elif top_id in val_alts[i]:
            scores.append(2)
        else:
            scores.append(1)

    return float(np.mean(scores))

# ── SEARCH ────────────────────────────────────────────────────────────────────
np.random.seed(RANDOM_SEED)
best_mhr    = -1.0
best_config = None
all_results = []

lam_values    = LAMBDA_VALUES
total_configs = len(lam_values) * N_WEIGHT_SAMPLES

print(f"Starting search: {total_configs} configurations...")
print(f"  {len(lam_values)} λ values × {N_WEIGHT_SAMPLES} Dirichlet samples")
print(f"  Validation: {len(VALIDATION_SET)} queries | Corpus: {len(phonex)} exercises")
print()

for lam in lam_values:
    weight_samples = np.random.dirichlet([1, 1, 1], size=N_WEIGHT_SAMPLES)
    for alpha, beta1, beta2 in weight_samples:
        mhr = compute_mhr_phonex(alpha, beta1, beta2, lam)
        all_results.append((mhr, alpha, beta1, beta2, lam))
        #if mhr > best_mhr:
        #    best_mhr    = mhr
        #    best_config = (alpha, beta1, beta2, lam)
        all_results.append((mhr, alpha, beta1, beta2, lam))
        # Do not update best_config here — select after sorting
    print(f"  λ={lam:.1f} | best MHR so far: {best_mhr:.3f}")

# Select best configuration among all that achieve max MHR
# Prefer highest α (cosine-dominant) among tied configurations
# Select best among all optimal configurations (highest α)
all_results.sort(reverse=True)
max_mhr = all_results[0][0]
optimal = [(mhr,a,b1,b2,l) for mhr,a,b1,b2,l in all_results if mhr == max_mhr]
best = max(optimal, key=lambda x: x[1])  # highest α
best_mhr = best[0]
alpha, beta1, beta2, lam = best[1], best[2], best[3], best[4]

# ── RESULTS ───────────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("SEARCH COMPLETE")
print("=" * 55)

print(f"\nBest configuration:")
print(f"  α  (cosine)       = {alpha:.4f}")
print(f"  β1 (C_DIFF)       = {beta1:.4f}")
print(f"  β2 (C_ERR)        = {beta2:.4f}")
print(f"  λ  (C_DIFF sharp) = {lam:.1f}")
print(f"  MHR               = {best_mhr:.3f} / 3.000")
print(f"  Check α+β1+β2     = {alpha+beta1+beta2:.6f}")

all_results.sort(reverse=True)
print()
print("Top 10 configurations:")
print(f"{'Rank':<5} {'MHR':>5} {'α':>7} {'β1':>7} {'β2':>7} {'λ':>5}")
print("-" * 40)
for rank, (mhr, a, b1, b2, l) in enumerate(all_results[:10], 1):
    print(f"{rank:<5} {mhr:>5.3f} {a:>7.3f} {b1:>7.3f} {b2:>7.3f} {l:>5.1f}")

mhr_vals = np.array([r[0] for r in all_results])
print()
print(f"MHR distribution: min={mhr_vals.min():.3f} "
      f"max={mhr_vals.max():.3f} mean={mhr_vals.mean():.3f}")

print()
print("Convergence check:")
for n in [20, 40, 60, 80, 100, 150, 200]:
    best_at_n = []
    for li, lam_val in enumerate(lam_values):
        start  = li * N_WEIGHT_SAMPLES
        subset = all_results[start:start+n]
        if subset:
            best_at_n.append(max(r[0] for r in subset))
    if best_at_n:
        print(f"  n={n:>3}: {np.mean(best_at_n):.3f}")

# Save
best_params = {
    'alpha': float(alpha), 'beta1': float(beta1),
    'beta2': float(beta2), 'lambda': float(lam),
    'mhr': float(best_mhr),
    'n_configs_searched': total_configs,
    'n_validation_queries': len(VALIDATION_SET),
    'corpus': 'PhonEx_v1.0', 'corpus_size': len(phonex),
}
with open('best_phonex_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print()
print("Saved best_phonex_params.json")
print("Update ALPHA_P, BETA1_P, BETA2_P, LAM_P in Cell 7 with these values.")



In [ ]:
# =============================================================================
# CELL — TF-IDF + BM25 Baselines + Ablation Study: Instantiation 2 (PhonEx)
# STANDALONE — does not require any previous cell to have been run
#              except: Cell 3 (model) and corpus JSON file on Drive
#
# What this cell does:
#   1. Loads PhonEx corpus from JSON
#   2. Encodes 20 validation queries using MiniLM
#   3. Evaluates TF-IDF baseline (lexical, simple word-frequency weighting)
#   4. Evaluates BM25 baseline (lexical, saturation + length normalization)
#   5. Evaluates 3 CASS ablation variants (cosine-only, C_DIFF only, C_ERR only)
#   6. Evaluates Full CASS (α=0.305, β1=0.186, β2=0.510, λ=0.2)
#   7. Prints complete ablation table with MHR and P@1
#
# Baselines comparison:
#   TF-IDF — simplest lexical baseline, no saturation or length correction
#   BM25   — stronger lexical baseline, adds saturation + length normalization
#   Unlike QESC, BM25 partially works here (MHR=1.450) because parent queries
#   and exercise descriptions share

import json
with open('PHONICS_CORPUS_v1.0.json', encoding='utf-8') as f:
    corpus = json.load(f)

# ── PARAMETERS ────────────────────────────────────────────────────────────────
A_P, B1_P, B2_P, L_P = 0.3047, 0.1855, 0.5099, 0.2

# ── VALIDATION SET (20 queries — copied from Cell 11) ────────────────────────
VALIDATION_SET = [
    ("Mon enfant confond b et d quand il lit des mots simples",
     "ex_vc_003", ["ex_vc_012", "ex_vc_011"]),
    ("Elle ne peut pas fusionner les sons pour lire un mot",
     "ex_bd_003", ["ex_bd_001", "ex_bd_012"]),
    ("Il saute des syllabes dans les mots de trois syllabes",
     "ex_so_003", ["ex_so_012", "ex_so_005"]),
    ("My child confuses b and d when reading simple words",
     "ex_vc_003", ["ex_vc_012", "ex_vc_002"]),
    ("Child reads letter by letter and cannot blend sounds together",
     "ex_bd_005", ["ex_bd_013", "ex_bd_003"]),
    ("My daughter reads bateau as dateau and skips the middle part",
     "ex_so_003", ["ex_so_001", "ex_so_012"]),
    ("He reads every word backwards and gets very frustrated",
     "ex_lr_001", ["ex_lr_010", "ex_lr_011"]),
    ("She reads sentences very slowly and loses the meaning completely",
     "ex_bd_007", ["ex_bd_016", "ex_bd_019"]),
    ("Every time he sees the letter b he writes d instead",
     "ex_vc_001", ["ex_vc_011", "ex_vc_002"]),
    ("She reads mou instead of mot and confuses similar vowel sounds",
     "ex_vs_001", ["ex_vs_002", "ex_vs_010"]),
    ("Child cannot distinguish b from d even with isolated letters",
     "ex_vc_001", ["ex_vc_002", "ex_vc_010"]),
    ("Child reads paragraphs but completely loses meaning while decoding",
     "ex_bd_009", ["ex_bd_007", "ex_bd_018"]),
    ("Mon fils lit correctement les mots seuls mais pas dans les phrases",
     "ex_bd_005", ["ex_so_006", "ex_bd_007"]),
    ("Weldi ma iqrach mezyan, kayqra harf harf",
     "ex_bd_001", ["ex_bd_003", "ex_bd_011"]),
    ("Benti katkhelt bin les syllabes, katsal les mots",
     "ex_so_001", ["ex_so_003", "ex_so_011"]),
    ("Ma iqrach les mots, kaykhelt les lettres",
     "ex_vc_001", ["ex_bd_001", "ex_lr_001"]),
    ("My child skips syllables and also confuses b and d",
     "ex_mx_001", ["ex_so_003", "ex_vc_003"]),
    ("Il lit à voix haute devant la classe mais fait beaucoup d erreurs",
     "ex_so_016", ["ex_bd_007", "ex_vc_007"]),
    ("Elle lit tou au lieu de tu et confond les voyelles proches",
     "ex_vs_004", ["ex_vs_001", "ex_vs_013"]),
    ("Child writes and reads letters in the wrong order constantly",
     "ex_lr_001", ["ex_lr_010", "ex_lr_003"]),
]

VAL_DIFF = [
    2.0, 2.0, 2.0, 2.0, 2.0,
    2.0, 2.0, 4.0, 2.0, 2.0,
    1.0, 5.0, 3.0, 2.0, 2.0,
    2.0, 2.0, 4.0, 2.0, 2.0,
]

VAL_ERR = [
    'visual_confusion', 'blending_difficulty', 'syllable_omission',
    'visual_confusion', 'blending_difficulty', 'syllable_omission',
    'letter_reversal',  'blending_difficulty', 'visual_confusion',
    'vowel_substitution', 'visual_confusion',  'blending_difficulty',
    'blending_difficulty', 'blending_difficulty', 'syllable_omission',
    'visual_confusion', 'syllable_omission',   'syllable_omission',
    'vowel_substitution', 'letter_reversal',
]

val_best = [v[1] for v in VALIDATION_SET]
val_alts = [v[2] for v in VALIDATION_SET]

assert len(VAL_DIFF) == len(VALIDATION_SET) == 20
assert len(VAL_ERR)  == len(VALIDATION_SET) == 20

# ── ENCODE VALIDATION QUERIES ─────────────────────────────────────────────────
print(f"Encoding {len(VALIDATION_SET)} validation queries...")
val_vecs = model.encode(
    [v[0] for v in VALIDATION_SET],
    normalize_embeddings=True
)
cos_matrix = val_vecs @ phonex_embeddings.T
diff_array_p = np.array([e['DIFF'] for e in phonex], dtype=float)
print(f"Cosine matrix: {cos_matrix.shape}")
print()


In [ ]:
# =============================================================================
# CELL 12 — BM25 Baseline + Ablation Study: Instantiation 2 (PhonEx)
# STANDALONE — does not require Cell 11 (weight search) to have been run
#
# Requires only: phonex, phonex_embeddings, model, C_ERR from Cells 4-5
# =============================================================================
!pip install rank_bm25 -q

import numpy as np
from rank_bm25 import BM25Okapi
# If not installed: !pip install rank_bm25 -q


# ── BM25 BASELINE ─────────────────────────────────────────────────────────────
tokenized_p = [e['description'].lower().split() for e in phonex]
bm25_p = BM25Okapi(tokenized_p)

def bm25_score_p():
    scores = []
    for i in range(len(VALIDATION_SET)):
        q_tokens = VALIDATION_SET[i][0].lower().split()
        bm25_scores = bm25_p.get_scores(q_tokens)
        top_id = phonex[int(np.argmax(bm25_scores))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    return float(np.mean(scores)), sum(1 for s in scores if s==3)/len(scores)

# ── CASS VARIANTS ─────────────────────────────────────────────────────────────

def score_variant_p(alpha, beta1, beta2, lam=L_P):
    scores = []
    for i in range(len(VALIDATION_SET)):
        cos    = cos_matrix[i]
        c_diff = np.exp(-lam * (VAL_DIFF[i] - diff_array_p) ** 2)
        c_err  = np.array([C_ERR(VAL_ERR[i], e['ERR_TYPE']) for e in phonex])
        cass   = alpha * cos + beta1 * c_diff + beta2 * c_err
        top_id = phonex[int(np.argmax(cass))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    return float(np.mean(scores)), sum(1 for s in scores if s==3)/len(scores)

# ── RUN ALL VARIANTS ──────────────────────────────────────────────────────────

print("=" * 55)
print("ABLATION STUDY — INSTANTIATION 2 (PhonEx)")
print(f"Validation set: {len(VALIDATION_SET)} queries")
print("=" * 55)
print()
print(f"{'System':<32} {'MHR':>6} {'P@1':>6}")
print("-" * 46)

variants_p = [
    ("BM25 baseline",                None),
    ("Cosine-only (α=1)",            (1.0,    0.0,        0.0      )),
    ("CASS + C_DIFF only (β₂=0)",    (A_P,    B1_P+B2_P,  0.0      )),
    ("CASS + C_ERR only (β₁=0)",     (A_P,    0.0,        B1_P+B2_P)),
    ("Full CASS (C_DIFF + C_ERR)",   (A_P,    B1_P,       B2_P     )),
]

all_results_p = {}
for name, params in variants_p:
    if params is None:
        mhr, p1 = bm25_score_p()
    else:
        mhr, p1 = score_variant_p(*params)
    all_results_p[name] = (mhr, p1)
    marker = " ←" if name == "Full CASS (C_DIFF + C_ERR)" else ""
    print(f"{name:<32} {mhr:>6.3f} {p1:>6.2f}{marker}")

print()
print("Key findings:")
bm25_mhr = all_results_p["BM25 baseline"][0]
cos_mhr  = all_results_p["Cosine-only (α=1)"][0]
cass_mhr = all_results_p["Full CASS (C_DIFF + C_ERR)"][0]

print(f"  Full CASS vs BM25:         ΔMHR={cass_mhr-bm25_mhr:+.3f}")
print(f"  Full CASS vs cosine-only:  ΔMHR={cass_mhr-cos_mhr:+.3f}")
print(f"  C_DIFF contribution:       ΔMHR={all_results_p['CASS + C_DIFF only (β₂=0)'][0]-cos_mhr:+.3f}")
print(f"  C_ERR contribution:        ΔMHR={all_results_p['CASS + C_ERR only (β₁=0)'][0]-cos_mhr:+.3f}")
print(f"  Synergy (both > each):     {cass_mhr:.3f} > {all_results_p['CASS + C_DIFF only (β₂=0)'][0]:.3f} and {all_results_p['CASS + C_ERR only (β₁=0)'][0]:.3f}")

In [ ]:
# =============================================================================
# CELL 13 — TF-IDF Baseline: Instantiation 2 (PhonEx)
# =============================================================================
# Computes TF-IDF retrieval score on the 20-query PhonEx validation set.
# TF-IDF indexes exercise descriptions and ranks by word-frequency similarity
# to parent/teacher queries.
# Unlike BM25, TF-IDF does not apply saturation or length normalization.
# Expected to partially work on direct French phonics queries (shared
# technical vocabulary) but fail on Darija and indirect descriptions.
#
# Requires: phonex, VALIDATION_SET, val_best, val_alts from previous cells
# Runtime: <5 seconds on CPU
# =============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build TF-IDF index on PhonEx exercise descriptions
corpus_texts = [e['description'] for e in phonex]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_texts)

def tfidf_score():
    scores = []
    for i in range(len(VALIDATION_SET)):
        q_vec = vectorizer.transform([VALIDATION_SET[i][0]])
        sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
        top_id = phonex[int(np.argmax(sims))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    mhr = float(np.mean(scores))
    p1  = sum(1 for s in scores if s==3)/len(scores)
    return mhr, p1

mhr_tfidf, p1_tfidf = tfidf_score()
print(f"TF-IDF baseline: MHR={mhr_tfidf:.3f}, P@1={p1_tfidf:.2f}")